In [0]:
%sql
CREATE CATALOG IF NOT EXISTS `virtue_foundation_silver`
COMMENT 'Silver layer: Cleaned, validated, and structured healthcare facility data';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`
COMMENT 'Cleaned and normalized healthcare facility data with parsed capabilities';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean` (
  -- Primary identifier
  `facility_id` STRING NOT NULL COMMENT 'Unique facility identifier (deduplicated)',
  
  -- Basic information
  `name` STRING COMMENT 'Facility name',
  `organization_type` STRING COMMENT 'Type of healthcare organization',
  `facility_type_id` STRING COMMENT 'Facility type classification ID',
  `operator_type_id` STRING COMMENT 'Operator type classification ID',
  `affiliation_type_ids` STRING COMMENT 'Affiliation type IDs',
  `description` STRING COMMENT 'Facility description',
  
  -- Address fields
  `address_line1` STRING COMMENT 'Primary address line',
  `address_line2` STRING COMMENT 'Secondary address line',
  `address_line3` STRING COMMENT 'Tertiary address line',
  `address_city` STRING COMMENT 'City name',
  `address_state` STRING COMMENT 'State or region name',
  `address_pincode` STRING COMMENT 'Cleaned pincode (spaces removed)',
  `address_country` STRING COMMENT 'Country',
  `address_country_code` STRING COMMENT 'Country code',
  
  -- Geographic coordinates
  `latitude` DOUBLE COMMENT 'Latitude coordinate (validated within India bounds)',
  `longitude` DOUBLE COMMENT 'Longitude coordinate (validated within India bounds)',
  `coordinates_geojson` STRING COMMENT 'GeoJSON point representation',
  
  -- Contact information
  `official_phone` STRING COMMENT 'Official phone number',
  `phone_numbers` ARRAY<STRING> COMMENT 'Array of all phone numbers',
  `email` STRING COMMENT 'Email address',
  `official_website` STRING COMMENT 'Official website URL',
  `websites` ARRAY<STRING> COMMENT 'Array of all website URLs',
  `facebook_link` STRING COMMENT 'Facebook page URL',
  
  -- Capacity metrics (cleaned to numeric)
  `number_doctors` INT COMMENT 'Number of doctors (parsed from text)',
  `number_doctors_raw` STRING COMMENT 'Original raw value',
  `capacity_beds` INT COMMENT 'Bed capacity (parsed from text)',
  `capacity_beds_raw` STRING COMMENT 'Original raw value',
  `area_sqft` DOUBLE COMMENT 'Facility area in square feet',
  
  -- Temporal fields
  `year_established` INT COMMENT 'Year facility was established',
  `recency_of_page_update` DATE COMMENT 'Last page update date',
  
  -- Digital presence metrics
  `distinct_social_media_presence_count` INT COMMENT 'Count of distinct social media platforms',
  `affiliated_staff_presence` STRING COMMENT 'Affiliated staff presence indicator',
  `custom_logo_presence` BOOLEAN COMMENT 'Whether facility has custom logo',
  `number_of_facts` INT COMMENT 'Number of facts about the organization',
  `most_recent_social_post_date` DATE COMMENT 'Most recent social media post date',
  `post_count` INT COMMENT 'Total social media post count',
  `follower_count` INT COMMENT 'Social media follower count',
  `likes_count` INT COMMENT 'Social media likes count',
  `engagement_count` INT COMMENT 'Social media engagement count',
  
  -- Source tracking
  `source` STRING COMMENT 'Primary data source',
  `source_types` ARRAY<STRING> COMMENT 'Array of source types',
  `source_ids` ARRAY<STRING> COMMENT 'Array of source IDs',
  `source_urls` ARRAY<STRING> COMMENT 'Array of source URLs',
  `cluster_id` STRING COMMENT 'Cluster identifier for entity resolution',
  
  -- Data quality metadata
  `data_quality_score` DOUBLE COMMENT 'Overall data quality score (0-100)',
  `has_valid_coordinates` BOOLEAN COMMENT 'Whether coordinates are valid',
  `has_valid_pincode` BOOLEAN COMMENT 'Whether pincode is valid',
  `completeness_score` DOUBLE COMMENT 'Field completeness score (0-100)',
  
  -- Audit fields
  `bronze_unique_id` STRING COMMENT 'Original unique_id from bronze layer',
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  `updated_at` TIMESTAMP COMMENT 'Record last update timestamp',
  `created_by` STRING COMMENT 'User/process that created record',
  
  CONSTRAINT facilities_clean_pk PRIMARY KEY (facility_id)
)
USING DELTA
CLUSTER BY (address_state, address_city, facility_type_id)
COMMENT 'Core cleaned facility data with validated fields and quality metrics'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.tuneFileSizesForRewrites' = 'true'
);
/*Clustering Rationale: State → City → Facility Type optimizes for:
Medical Desert Planner (geographic queries)
Facility searches by location and type
Regional analytics
3.2 facility_specialties (Normalized)*/
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facility_specialties` (
  `specialty_id` STRING NOT NULL COMMENT 'Unique specialty record identifier',
  `facility_id` STRING NOT NULL COMMENT 'Foreign key to facilities_clean',
  `specialty_code` STRING COMMENT 'Standardized specialty code (e.g., ophthalmology, cardiology)',
  `specialty_name` STRING COMMENT 'Human-readable specialty name',
  `specialty_category` STRING COMMENT 'Specialty category (e.g., surgical, diagnostic, primary_care)',
  `confidence_score` DOUBLE COMMENT 'Extraction confidence score (0-1)',
  `source_text` STRING COMMENT 'Original text from which specialty was extracted',
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  
  CONSTRAINT facility_specialties_pk PRIMARY KEY (specialty_id),
  CONSTRAINT facility_specialties_fk FOREIGN KEY (facility_id) REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (specialty_code, facility_id)
COMMENT 'Parsed and normalized facility specialties (one row per facility-specialty)'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
/*Clustering Rationale: Specialty Code → Facility ID optimizes for:
Referral Copilot (finding facilities by specialty)
Specialty-based searches and filtering
3.3 facility_equipment (Normalized)*/
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facility_equipment` (
  `equipment_id` STRING NOT NULL COMMENT 'Unique equipment record identifier',
  `facility_id` STRING NOT NULL COMMENT 'Foreign key to facilities_clean',
  `equipment_type` STRING COMMENT 'Equipment type (e.g., imaging, surgical, diagnostic)',
  `equipment_name` STRING COMMENT 'Equipment name (e.g., MRI scanner, CT scanner)',
  `equipment_category` STRING COMMENT 'High-level category (e.g., radiology, laboratory)',
  `quantity` INT COMMENT 'Number of units (if specified)',
  `confidence_score` DOUBLE COMMENT 'Extraction confidence score (0-1)',
  `source_text` STRING COMMENT 'Original text from which equipment was extracted',
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  
  CONSTRAINT facility_equipment_pk PRIMARY KEY (equipment_id),
  CONSTRAINT facility_equipment_fk FOREIGN KEY (facility_id) REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (equipment_category, equipment_name, facility_id)
COMMENT 'Parsed and normalized facility equipment (one row per facility-equipment)'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
/*Clustering Rationale: Equipment Category → Name → Facility ID optimizes for:
Referral Copilot (finding facilities with specific equipment)
Equipment-based capability searches
3.4 facility_procedures (Normalized)*/
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facility_procedures` (
  `procedure_id` STRING NOT NULL COMMENT 'Unique procedure record identifier',
  `facility_id` STRING NOT NULL COMMENT 'Foreign key to facilities_clean',
  `procedure_code` STRING COMMENT 'Standardized procedure code',
  `procedure_name` STRING COMMENT 'Procedure name',
  `procedure_category` STRING COMMENT 'Procedure category (e.g., surgical, diagnostic, therapeutic)',
  `confidence_score` DOUBLE COMMENT 'Extraction confidence score (0-1)',
  `source_text` STRING COMMENT 'Original text from which procedure was extracted',
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  
  CONSTRAINT facility_procedures_pk PRIMARY KEY (procedure_id),
  CONSTRAINT facility_procedures_fk FOREIGN KEY (facility_id) REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (procedure_category, procedure_code, facility_id)
COMMENT 'Parsed and normalized facility procedures (one row per facility-procedure)'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
/*Clustering Rationale: Procedure Category → Code → Facility ID optimizes for:
Referral Copilot (finding facilities offering specific procedures)
Procedure-based capability searches
3.5 facility_capabilities (Normalized)*/
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facility_capabilities` (
  `capability_id` STRING NOT NULL COMMENT 'Unique capability record identifier',
  `facility_id` STRING NOT NULL COMMENT 'Foreign key to facilities_clean',
  `capability_type` STRING COMMENT 'Capability type (e.g., certification, accreditation, service)',
  `capability_description` STRING COMMENT 'Capability description',
  `capability_category` STRING COMMENT 'High-level category',
  `confidence_score` DOUBLE COMMENT 'Extraction confidence score (0-1)',
  `source_text` STRING COMMENT 'Original text from which capability was extracted',
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  
  CONSTRAINT facility_capabilities_pk PRIMARY KEY (capability_id),
  CONSTRAINT facility_capabilities_fk FOREIGN KEY (facility_id) REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (capability_category, capability_type, facility_id)
COMMENT 'Parsed and normalized facility capabilities (one row per facility-capability)'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
/*Clustering Rationale: Capability Category → Type → Facility ID optimizes for:
Facility Trust Desk (finding certified/accredited facilities)
Capability-based filtering
3.6 data_quality_flags (Quality Tracking)*/
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`data_quality_flags` (
  `flag_id` STRING NOT NULL COMMENT 'Unique flag identifier',
  `facility_id` STRING NOT NULL COMMENT 'Foreign key to facilities_clean',
  `flag_type` STRING COMMENT 'Type of quality issue (e.g., missing_coords, invalid_pincode, stale_data)',
  `flag_severity` STRING COMMENT 'Severity level (critical, high, medium, low)',
  `flag_description` STRING COMMENT 'Detailed description of the issue',
  `flagged_at` TIMESTAMP COMMENT 'When the flag was raised',
  `resolved_at` TIMESTAMP COMMENT 'When the flag was resolved (NULL if unresolved)',
  `resolved_by` STRING COMMENT 'User/process that resolved the flag',
  
  CONSTRAINT data_quality_flags_pk PRIMARY KEY (flag_id),
  CONSTRAINT data_quality_flags_fk FOREIGN KEY (facility_id) REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (flag_severity, flag_type, facility_id)
COMMENT 'Data quality flags and issues per facility'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
/*Clustering Rationale: Severity → Type → Facility ID optimizes for:
Data Readiness Desk (prioritizing critical issues)
Quality monitoring dashboards
3.7 transformation_audit_log (Lineage Tracking)*/
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`transformation_audit_log` (
  `log_id` STRING NOT NULL COMMENT 'Unique log entry identifier',
  `facility_id` STRING COMMENT 'Facility affected (NULL for batch operations)',
  `transformation_type` STRING COMMENT 'Type of transformation (deduplication, parsing, validation, etc.)',
  `transformation_status` STRING COMMENT 'Status (success, failure, partial)',
  `records_processed` INT COMMENT 'Number of records processed',
  `records_succeeded` INT COMMENT 'Number of records successfully transformed',
  `records_failed` INT COMMENT 'Number of records that failed',
  `error_message` STRING COMMENT 'Error message if transformation failed',
  `execution_time_seconds` DOUBLE COMMENT 'Execution time in seconds',
  `executed_by` STRING COMMENT 'User/process that executed transformation',
  `executed_at` TIMESTAMP COMMENT 'Execution timestamp',
  
  CONSTRAINT transformation_audit_log_pk PRIMARY KEY (log_id)
)
USING DELTA
CLUSTER BY (executed_at, transformation_type, transformation_status)
COMMENT 'Audit log for all Bronze → Silver transformations'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
/*Clustering Rationale: Execution Time → Type → Status optimizes for:
Temporal audit queries (recent transformations)
Monitoring transformation success rates
Key Benefits of Liquid Clustering
Automatic Optimization: Databricks automatically maintains optimal data layout as data grows
No Manual OPTIMIZE Required: Clustering happens incrementally during writes
Query Performance: 3-10x faster queries on clustered columns
Flexible: Can change clustering keys without rewriting entire table
Cost Efficient: Reduces data scanning and compute costs
Performance Expectations by App
Medical Desert Planner: Geographic queries (state/city filters) will be 5-10x faster
Referral Copilot: Specialty/equipment/procedure lookups will be 3-5x faster
Facility Trust Desk: Quality flag queries will be 4-8x faster
Data Readiness Desk: Audit log queries will be 2-4x faster
Execution Instructions
Open Databricks SQL Editor or Notebook with write permissions
Execute scripts in order (Steps 1-3)
Verify clustering is enabled:
DESCRIBE EXTENDED virtue_foundation_silver.healthcare_facilities.facilities_clean;
-- Look for "Clustering Information" in output
Monitoring Liquid Clustering
After data is loaded, monitor clustering effectiveness:
-- Check clustering statistics
DESCRIBE DETAIL virtue_foundation_silver.healthcare_facilities.facilities_clean;

-- View clustering columns
SHOW TBLPROPERTIES virtue_foundation_silver.healthcare_facilities.facilities_clean;
The clustering will automatically optimize as you insert/update data. No manual maintenance required!
Would you like me to help design the ETL pipeline to populate these tables from Bronze, or provide sample queries optimized for the clustering strategy?
Is this useful?
*/



In [0]:
%sql
-- Create enriched address table
-- Note: Foreign key constraints removed temporarily to avoid reference issues
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facility_address_enriched` (
  `facility_id` STRING NOT NULL,
  `validated_pincode` BIGINT COMMENT 'Validated pincode from India Post directory',
  `pincode_office_name` STRING COMMENT 'Post office name',
  `pincode_office_type` STRING COMMENT 'HO/BO/PO',
  `validated_district` STRING COMMENT 'Official district name from pincode directory',
  `validated_state` STRING COMMENT 'Official state name from pincode directory',
  `region_name` STRING COMMENT 'Postal region',
  `circle_name` STRING COMMENT 'Postal circle',
  `pincode_latitude` STRING COMMENT 'Pincode centroid latitude',
  `pincode_longitude` STRING COMMENT 'Pincode centroid longitude',
  `address_validation_status` STRING COMMENT 'valid/invalid/partial',
  `validation_confidence` DOUBLE COMMENT 'Confidence score (0-1)',
  `created_at` TIMESTAMP,
  
  CONSTRAINT facility_address_enriched_pk PRIMARY KEY (facility_id)
  -- FOREIGN KEY constraint will be added after ensuring parent table has PRIMARY KEY
  -- CONSTRAINT facility_address_enriched_fk FOREIGN KEY (facility_id) 
  --   REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (validated_state, validated_district, validated_pincode)
COMMENT 'Facility addresses enriched and validated with India Post pincode directory';
--9. facility_district_health_context (New)
CREATE TABLE IF NOT EXISTS `virtue_foundation_silver`.`healthcare_facilities`.`facility_district_health_context` (
  `facility_id` STRING NOT NULL,
  `district_name` STRING COMMENT 'District name from NFHS-5',
  `state_ut` STRING COMMENT 'State/UT from NFHS-5',
  
  -- Population demographics
  `households_surveyed` DOUBLE,
  `population_below_age_15_years_pct` DOUBLE,
  `sex_ratio_total_f_per_1000_m` DOUBLE,
  
  -- Health infrastructure indicators
  `institutional_birth_5y_pct` DOUBLE,
  `institutional_birth_in_public_facility_5y_pct` DOUBLE,
  
  -- Health outcomes
  `child_u5_who_are_stunted_pct` STRING,
  `child_u5_who_are_underweight_pct` STRING,
  `non_pregnant_w15_49_who_are_anaemic_pct` DOUBLE,
  
  -- Healthcare access
  `mothers_who_had_at_least_4_anc_visits_lb5y_pct` STRING,
  `child_12_23m_fully_vaccinated_pct` STRING,
  
  -- Sanitation & infrastructure
  `hh_electricity_pct` DOUBLE,
  `hh_improved_water_pct` DOUBLE,
  `hh_use_improved_sanitation_pct` DOUBLE,
  `households_using_clean_fuel_for_cooking_pct` DOUBLE,
  
  -- Health insurance
  `hh_member_covered_health_insurance_pct` DOUBLE,
  
  `created_at` TIMESTAMP,
  
  CONSTRAINT facility_district_health_context_pk PRIMARY KEY (facility_id)
  -- FOREIGN KEY constraint will be added after ensuring parent table has PRIMARY KEY
  -- CONSTRAINT facility_district_health_context_fk FOREIGN KEY (facility_id) 
  --   REFERENCES `virtue_foundation_silver`.`healthcare_facilities`.`facilities_clean`(facility_id)
)
USING DELTA
CLUSTER BY (state_ut, district_name)
COMMENT 'District-level health indicators from NFHS-5 linked to facilities';



